# 03 — SciPy — traitement du signal

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- générer des signaux et calculer leur FFT ;
- concevoir et appliquer des filtres IIR (Butterworth) et FIR ;
- détecter des pics avec `find_peaks` ;
- convoluer et corréler des signaux ;
- rééchantillonner avec `resample` et `decimate` ;
- estimer la densité spectrale avec Welch et le spectrogramme.

## Prérequis — ce que vous devez maîtriser

À ce stade de la formation, vous savez :

- NumPy (arrays, fonctions mathématiques) ;
- notions de fréquence, période, Fourier ;
- matplotlib pour la visualisation.

Ce que ce notebook **ne couvrira pas** :

- l’analyse musicale complète (librosa) ;
- les télécoms et radars avancés.

## Plan

1. Installation et imports
2. Générer un signal synthétique
3. FFT — Transformée de Fourier rapide
4. Spectre d’amplitude et de phase
5. Filtres IIR (Butterworth)
6. Filtres FIR
7. Détection de pics (`find_peaks`)
8. Convolution et corrélation
9. Rééchantillonnage
10. Estimation spectrale (Welch)
11. Spectrogramme
12. Synthèse
13. Exercices
14. Ressources externes

### Exemple : analyse spectrale d'un signal musical

Créons un signal qui simule un accord musical (Do majeur : C4, E4, G4)
et analysons son spectre.

In [ ]:
import numpy as np
from scipy.signal import find_peaks, butter, sosfilt
from scipy.fft import fft, fftfreq
import matplotlib.pyplot as plt

# Accord Do majeur
fs = 8000  # Hz
t = np.linspace(0, 1, fs)
freqs = [261.63, 329.63, 392.00]  # C4, E4, G4
signal = sum(np.sin(2*np.pi*f*t) for f in freqs)
signal += 0.1 * np.random.randn(len(t))  # bruit

# FFT
N = len(signal)
yf = np.abs(fft(signal))[:N//2]
xf = fftfreq(N, 1/fs)[:N//2]

# Détection des pics
peaks, props = find_peaks(yf, height=50, distance=50)
print('Fréquences détectées :', xf[peaks][:5])

plt.figure(figsize=(10, 4))
plt.plot(xf[:2000], yf[:2000])
plt.plot(xf[peaks], yf[peaks], 'rv', ms=10)
plt.xlabel('Fréquence (Hz)'); plt.ylabel('Amplitude')
plt.title('Spectre de l\'accord Do majeur')
plt.grid(); plt.show()

### Exercice : filtre passe-bande

À partir du signal musical ci-dessus :

1. Concevez un filtre passe-bande Butterworth (300-350 Hz) pour isoler la note E4
2. Appliquez le filtre et tracez le signal filtré
3. Vérifiez avec la FFT que seule la fréquence ~330 Hz reste

In [ ]:
# Filtre passe-bande pour isoler E4
low, high = 300, 350
sos = butter(4, [low, high], btype='bandpass', fs=fs, output='sos')
filtered = sosfilt(sos, signal)

# Vérification spectrale
yf_filt = np.abs(fft(filtered))[:N//2]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
ax1.plot(t[:500], filtered[:500])
ax1.set_title('Signal filtré (E4 isolé)'); ax1.set_xlabel('Temps (s)')
ax2.plot(xf[:2000], yf_filt[:2000])
ax2.set_title('Spectre du signal filtré')
ax2.set_xlabel('Fréquence (Hz)')
plt.tight_layout(); plt.show()

### Exercice avancé : détection de QRS dans un ECG simulé

Un signal ECG contient des complexes QRS (pics marqués correspondant aux battements cardiaques).

1. Générez un ECG synthétique avec `scipy.misc.electrocardiogram()` (si disponible)
   ou créez un signal périodique avec des pics
2. Utilisez `find_peaks` avec les paramètres `height`, `distance` et `prominence`
3. Calculez la fréquence cardiaque moyenne à partir des intervalles R-R

In [ ]:
# ECG synthétique simplifié
fs_ecg = 360
t_ecg = np.linspace(0, 10, fs_ecg * 10)

# Simulation d'un ECG : pics réguliers + bruit
bpm = 72
period = 60 / bpm
ecg = np.zeros_like(t_ecg)
for beat_time in np.arange(0, 10, period):
    idx = np.argmin(np.abs(t_ecg - beat_time))
    if idx < len(ecg) - 5:
        ecg[idx:idx+5] = [0.1, 0.8, 1.5, -0.5, 0.1]  # forme QRS simplifiée
ecg += 0.05 * np.random.randn(len(ecg))

peaks_ecg, _ = find_peaks(ecg, height=0.5, distance=fs_ecg*0.5)
rr_intervals = np.diff(peaks_ecg) / fs_ecg
hr = 60 / np.mean(rr_intervals)
print(f'Fréquence cardiaque : {hr:.1f} BPM (attendu : {bpm})')

plt.figure(figsize=(12, 3))
plt.plot(t_ecg, ecg)
plt.plot(t_ecg[peaks_ecg], ecg[peaks_ecg], 'rv', ms=8)
plt.xlabel('Temps (s)'); plt.title('ECG synthétique')
plt.show()

---

## 1. Installation et imports

In [ ]:
import numpy as np
from scipy import signal
import scipy
print(f"SciPy {scipy.__version__}")

---

## 2. Générer un signal synthétique

In [ ]:
fs = 1000  # Hz (fréquence d'échantillonnage)
t = np.linspace(0, 1, fs, endpoint=False)  # 1 seconde

# Sinus 50 Hz + sinus 120 Hz
sig = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t)
print(f"Signal: {sig.shape[0]} échantillons, durée: {t[-1]:.3f} s")

In [ ]:
# Signal avec bruit
rng = np.random.default_rng(42)
sig_noisy = sig + 0.5 * rng.standard_normal(len(t))
print(f"SNR (signal/bruit) : {np.std(sig)/np.std(sig_noisy - sig):.2f}")

---

## 3. FFT — Transformée de Fourier rapide

La FFT décompose un signal temporel en ses composantes fréquentielles.

In [ ]:
fft_vals = np.fft.fft(sig)
freqs = np.fft.fftfreq(len(sig), 1 / fs)
amp = np.abs(fft_vals[:len(fft_vals) // 2])
freqs_pos = freqs[:len(freqs) // 2]

print(f"Pic principal à {freqs_pos[amp.argmax()]:.0f} Hz")

In [ ]:
# Les 5 plus grandes amplitudes
idx_sorted = np.argsort(amp)[::-1][:5]
for i in idx_sorted:
    print(f"{freqs_pos[i]:6.1f} Hz : amplitude = {amp[i]:.1f}")

---

## 4. Spectre d’amplitude et de phase

In [ ]:
phase = np.angle(fft_vals[:len(fft_vals) // 2])
print(f"Phase au pic 50 Hz: {phase[np.argmin(np.abs(freqs_pos - 50))]:.4f} rad")

### FFT et résolution fréquentielle

La résolution = `fs / N`. Plus le signal est long, plus la résolution est fine.

In [ ]:
N = len(sig)
resolution = fs / N
print(f"Résolution fréquentielle: {resolution} Hz")

---

## 5. Filtres IIR (Butterworth)

Les filtres IIR sont efficaces mais peuvent être instables à ordre élevé.

In [ ]:
from scipy.signal import butter, filtfilt, sosfiltfilt

# Passe-bas à 80 Hz, ordre 4
b, a = butter(4, 80, btype='low', fs=fs)
filtered = filtfilt(b, a, sig_noisy)
print(f"Échantillons filtrés: {len(filtered)}")

### `filtfilt` vs `lfilter`

- `lfilter` : filtrage causal (introduit un déphasage)
- `filtfilt` : filtrage zéro-phase (applique le filtre deux fois, aller-retour)

In [ ]:
# Passe-haut à 100 Hz
b, a = butter(4, 100, btype='high', fs=fs)
high_pass = filtfilt(b, a, sig_noisy)
print(f"Après passe-haut: énergie = {np.sum(high_pass**2):.2f}")

In [ ]:
# Passe-bande 40-60 Hz
b, a = butter(4, [40, 60], btype='band', fs=fs)
bandpass = filtfilt(b, a, sig_noisy)
print(f"Passe-bande 40-60 Hz: énergie = {np.sum(bandpass**2):.2f}")

### Utiliser les SOS (Second Order Sections)

Plus stable numériquement que la représentation `b, a`.

In [ ]:
sos = butter(6, 80, btype='low', fs=fs, output='sos')
filtered_sos = sosfiltfilt(sos, sig_noisy)
print(f"SOS filtré: {len(filtered_sos)} samples")

---

## 6. Filtres FIR

Les filtres FIR sont toujours stables et ont une phase linéaire.

In [ ]:
from scipy.signal import firwin, lfilter

taps = firwin(101, 80, fs=fs)  # 101 coefficients, fc = 80 Hz
fir_filtered = lfilter(taps, 1.0, sig_noisy)
print(f"FIR: {len(taps)} taps, délai = {(len(taps)-1)//2} samples")

In [ ]:
# FIR passe-bande
taps_bp = firwin(201, [40, 60], pass_zero=False, fs=fs)
fir_bp = lfilter(taps_bp, 1.0, sig_noisy)
print(f"FIR passe-bande: {len(taps_bp)} taps")

---

## 7. Détection de pics (`find_peaks`)

In [ ]:
from scipy.signal import find_peaks

peaks, properties = find_peaks(sig, height=0.8, distance=10)
print(f"Nombre de pics: {len(peaks)}")

In [ ]:
# Avec prominence
peaks, props = find_peaks(sig, prominence=0.5)
print(f"Pics prominents: {len(peaks)}")
print(f"Prominences: {props['prominences'][:5]}")

In [ ]:
# Largeur des pics
peaks, props = find_peaks(sig, width=3)
print(f"Largeurs: {props['widths'][:5]}")

---

## 8. Convolution et corrélation

In [ ]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([1, 0, -1])

# Convolution
print(f"Convolve (full):  {signal.convolve(a, b, mode='full')}")
print(f"Convolve (valid): {signal.convolve(a, b, mode='valid')}")
print(f"Convolve (same):  {signal.convolve(a, b, mode='same')}")

In [ ]:
# Corrélation croisée
print(f"Correlate: {signal.correlate(a, b, mode='valid')}")

### Convolution rapide (FFT)

In [ ]:
from scipy.signal import fftconvolve
big_a = rng.standard_normal(10_000)
big_b = rng.standard_normal(1_000)
result = fftconvolve(big_a, big_b, mode='valid')
print(f"FFT convolve: {result.shape}")  # beaucoup plus rapide pour les grands signaux

---

## 9. Rééchantillonnage

In [ ]:
# Resample de 1000 Hz à 500 Hz
resampled = signal.resample(sig, 500)
print(f"Original: {len(sig)}, Resample: {len(resampled)}")

In [ ]:
# Decimate : filtre anti-aliasing + sous-échantillonnage
decimated = signal.decimate(sig, 2)
print(f"Decimate par 2: {len(decimated)}")

---

## 10. Estimation spectrale (Welch)

Moyenne de FFT sur des fenêtres : moins de bruit spectral.

In [ ]:
f_welch, Pxx = signal.welch(sig_noisy, fs=fs, nperseg=256)
print(f"Pic Welch à {f_welch[Pxx.argmax()]:.1f} Hz")

In [ ]:
# Influence de nperseg
for nperseg in [64, 128, 256, 512]:
    f_w, P_w = signal.welch(sig_noisy, fs=fs, nperseg=nperseg)
    print(f"nperseg={nperseg:3d}: résolution={f_w[1]:.1f} Hz, pic={f_w[P_w.argmax()]:.1f} Hz")

---

## 11. Spectrogramme

Représentation temps-fréquence pour les signaux non stationnaires.

In [ ]:
# Signal à fréquence variable (chirp)
t_long = np.linspace(0, 2, 2000)
chirp = signal.chirp(t_long, f0=10, f1=200, t1=2)
f_spec, t_spec, Sxx = signal.spectrogram(chirp, fs=1000)
print(f"Spectro shape: {Sxx.shape}  (freq x time)")

---

## Synthèse

| Opération | Fonction | Notes |
|---|---|---|
| FFT | `np.fft.fft(sig)` | + `fftfreq(N, 1/fs)` |
| IIR Butter | `butter(N, fc, fs=)` + `filtfilt` | Zéro-phase |
| IIR SOS | `butter(..., output='sos')` + `sosfiltfilt` | Plus stable |
| FIR | `firwin(N, fc, fs=)` + `lfilter` | Phase linéaire |
| Pics | `find_peaks(sig, height=, distance=)` | + prominence, width |
| Convolution | `signal.convolve(a, b)` | full/valid/same |
| Corrélation | `signal.correlate(a, b)` | |
| Resample | `signal.resample(sig, N)` | |
| Decimate | `signal.decimate(sig, q)` | Filtre + ↓ |
| Welch | `signal.welch(sig, fs=)` | PSD |
| Spectro | `signal.spectrogram(sig, fs=)` | Temps-fréquence |

### Règles à retenir

- **`filtfilt`** = zéro-phase (filtre aller-retour).
- **Nyquist** : `fc < fs / 2`.
- **Welch** = FFT moyennée = moins de bruit spectral.
- **IIR rapide** mais peut être instable ; **FIR toujours stable**.
- **SOS** recommandé pour les filtres d’ordre > 4.

---

---

## Fenetrage

Le fenetrage reduit les fuites spectrales (spectral leakage) lors de la FFT.

In [ ]:
import numpy as np
from scipy.signal import windows

for name in ['hann', 'hamming', 'blackman', 'flattop']:
    w = windows.get_window(name, 256)
    print(f"{name:10s}: longueur={len(w)}, somme={w.sum():.2f}")

In [ ]:
# Appliquer une fenetre avant FFT
fs = 1000
t = np.linspace(0, 1, fs, endpoint=False)
sig = np.sin(2 * np.pi * 50 * t)
w = windows.hann(len(sig))
fft_no_win = np.abs(np.fft.fft(sig)[:500])
fft_win = np.abs(np.fft.fft(sig * w)[:500])
freqs = np.fft.fftfreq(len(sig), 1/fs)[:500]
print(f"Sans fenetre: pic={fft_no_win.max():.1f}")
print(f"Avec Hann:    pic={fft_win.max():.1f}")

---

## Chirp et signaux synthetiques

In [ ]:
from scipy import signal as sig_mod
import numpy as np

t = np.linspace(0, 1, 1000)

# Chirp lineaire 10 Hz -> 200 Hz
chirp = sig_mod.chirp(t, f0=10, f1=200, t1=1, method='linear')
print(f"Chirp: {len(chirp)} samples")

In [ ]:
# Signal carre
square = sig_mod.square(2 * np.pi * 5 * t)
print(f"Square: valeurs uniques = {set(square)}")

In [ ]:
# Dent de scie
saw = sig_mod.sawtooth(2 * np.pi * 5 * t)
print(f"Sawtooth: min={saw.min():.1f}, max={saw.max():.1f}")

## Exercices

### Exercice 1 *(facile)* — FFT d’un sinus pur

Échantillonnez un sinus 100 Hz à 1000 Hz pendant 1 s. Calculez la FFT et vérifiez que le pic est à 100 Hz.

In [ ]:
import numpy as np

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np

fs = 1000
t = np.linspace(0, 1, fs, endpoint=False)
sig = np.sin(2 * np.pi * 100 * t)
fft = np.fft.fft(sig)
freqs = np.fft.fftfreq(len(sig), 1 / fs)
amp = np.abs(fft[:len(fft)//2])
print(f"Pic à {freqs[:len(freqs)//2][amp.argmax()]:.0f} Hz")
```

</details>

### Exercice 2 *(facile)* — Détection de pics

Générez `sig = sin(2*pi*3*t) + bruit` pour `t` dans `[0, 2]` (500 points). Trouvez les pics avec `height=0.5`.

In [ ]:
import numpy as np
from scipy.signal import find_peaks

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np
from scipy.signal import find_peaks

rng = np.random.default_rng(0)
t = np.linspace(0, 2, 500)
sig = np.sin(2 * np.pi * 3 * t) + 0.1 * rng.standard_normal(500)
peaks, _ = find_peaks(sig, height=0.5)
print(f"{len(peaks)} pics trouvés")
```

</details>

### Exercice 3 *(moyen)* — Filtre passe-bas

Sur `sin(50 Hz) + sin(200 Hz)` à fs=1000, appliquez un Butterworth passe-bas à 100 Hz. Vérifiez que la composante 200 Hz est atténuée.

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np
from scipy.signal import butter, filtfilt

fs = 1000
t = np.linspace(0, 1, fs, endpoint=False)
sig = np.sin(2 * np.pi * 50 * t) + np.sin(2 * np.pi * 200 * t)
b, a = butter(4, 100, btype='low', fs=fs)
filt = filtfilt(b, a, sig)
fft_orig = np.abs(np.fft.fft(sig)[:500])
fft_filt = np.abs(np.fft.fft(filt)[:500])
freqs = np.fft.fftfreq(1000, 1/fs)[:500]
print(f"200 Hz avant: {fft_orig[200]:.1f}")
print(f"200 Hz après: {fft_filt[200]:.1f}")
```

</details>

### Exercice 4 *(moyen)* — Welch avec bruit

Générez 5 secondes de sinus 50 Hz + bruit fort (σ=1). Utilisez Welch pour retrouver le pic malgré le bruit.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np
from scipy import signal

fs = 1000
t = np.linspace(0, 5, 5 * fs)
sig = np.sin(2 * np.pi * 50 * t) + np.random.default_rng(0).standard_normal(len(t))
f, Pxx = signal.welch(sig, fs=fs, nperseg=1024)
print(f"Pic Welch : {f[Pxx.argmax()]:.1f} Hz")
```

</details>

### Exercice 5 *(difficile)* — Filtrage fréquentiel par FFT

Décomposez un signal avec FFT, mettez à zéro les composantes > 100 Hz, reconstruisez avec IFFT. Comparez l’énergie avant/après.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np

fs = 1000
t = np.linspace(0, 1, fs, endpoint=False)
sig = np.sin(2 * np.pi * 50 * t) + np.sin(2 * np.pi * 150 * t)
fft = np.fft.fft(sig)
freqs = np.fft.fftfreq(len(sig), 1 / fs)
fft[np.abs(freqs) > 100] = 0
reconstructed = np.fft.ifft(fft).real
print(f"Énergie avant: {np.sum(sig**2):.1f}")
print(f"Énergie après: {np.sum(reconstructed**2):.1f}")
```

</details>

### Approfondir : spectrogramme

Le spectrogramme montre comment le contenu fréquentiel évolue dans le temps.
C'est l'outil de référence pour l'analyse de signaux non stationnaires.

In [ ]:
from scipy.signal import spectrogram
import numpy as np
import matplotlib.pyplot as plt

# Signal chirp (fréquence qui augmente)
fs = 1000
t = np.linspace(0, 5, fs*5)
f0, f1 = 10, 200  # de 10 Hz à 200 Hz
chirp = np.sin(2*np.pi * (f0*t + (f1-f0)/(2*5) * t**2))

f, t_spec, Sxx = spectrogram(chirp, fs, nperseg=256)

plt.figure(figsize=(10, 4))
plt.pcolormesh(t_spec, f, 10*np.log10(Sxx+1e-10), shading='auto', cmap='magma')
plt.ylabel('Fréquence (Hz)'); plt.xlabel('Temps (s)')
plt.colorbar(label='dB'); plt.title('Spectrogramme d\'un chirp')
plt.ylim(0, 250); plt.show()

### Exercice : détection d'anomalies dans un signal

1. Générez un signal sinusoidal de 10 secondes à 1 Hz
2. Ajoutez 3 impulsions brèves (anomalies) à des instants aléatoires
3. Utilisez `find_peaks` avec `prominence` pour les détecter
4. Calculez le taux de détection

In [ ]:
import numpy as np
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

np.random.seed(42)
fs = 500
t = np.linspace(0, 10, fs*10)
signal = np.sin(2*np.pi*1*t)

# Anomalies
anomaly_times = sorted(np.random.choice(len(t)//2, 3, replace=False) * 2)
for idx in anomaly_times:
    signal[idx:idx+5] += 3  # impulsion

peaks, props = find_peaks(signal, prominence=2)
print(f'Anomalies injectées à : {[t[i] for i in anomaly_times]}')
print(f'Pics détectés à : {t[peaks].tolist()}')
print(f'Détection : {len(peaks)}/{len(anomaly_times)}')

plt.figure(figsize=(12, 3))
plt.plot(t, signal)
plt.plot(t[peaks], signal[peaks], 'rv', ms=10, label='détections')
plt.legend(); plt.grid(); plt.show()

### Exercice : conception de filtre coupe-bande

Le filtre coupe-bande (notch) supprime une fréquence précise.
Application typique : supprimer le 50 Hz du secteur dans un signal biomédical.

1. Créez un signal ECG simplifié contaminé par du 50 Hz
2. Concevez un filtre coupe-bande à 50 Hz avec `iirnotch`
3. Appliquez le filtre et comparez avant/après dans le domaine fréquentiel

In [ ]:
from scipy.signal import iirnotch, filtfilt
from scipy.fft import fft, fftfreq
import numpy as np
import matplotlib.pyplot as plt

fs = 500
t = np.linspace(0, 2, fs*2)
# Signal utile + bruit 50 Hz
ecg = np.sin(2*np.pi*1.2*t) * np.exp(-((t-1)**2)/0.5)
noise_50hz = 0.3 * np.sin(2*np.pi*50*t)
contaminated = ecg + noise_50hz

# Filtre notch à 50 Hz
b, a = iirnotch(50, 30, fs)  # freq, Q-factor, fs
cleaned = filtfilt(b, a, contaminated)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(t, contaminated, label='contaminé')
axes[0].plot(t, cleaned, label='filtré', alpha=0.8)
axes[0].legend(); axes[0].set_title('Domaine temporel')

# Spectre
N = len(t)
freq = fftfreq(N, 1/fs)[:N//2]
for sig, label in [(contaminated,'avant'), (cleaned,'après')]:
    spec = np.abs(fft(sig))[:N//2]
    axes[1].plot(freq, spec, label=label)
axes[1].set_xlim(0, 100); axes[1].legend()
axes[1].set_title('Spectre'); plt.tight_layout(); plt.show()

### Exercice bonus : analyse de fenêtrage

Le choix de la fenêtre (Hanning, Hamming, Blackman, etc.) affecte la résolution
spectrale et les fuites spectrales. Comparez visuellement.

In [ ]:
from scipy.signal import windows
from scipy.fft import fft, fftfreq
import numpy as np
import matplotlib.pyplot as plt

N = 256
fig, axes = plt.subplots(2, 1, figsize=(10, 6))

# Fenêtres dans le domaine temporel
for name, win_func in [('Rectangular', np.ones), ('Hanning', windows.hann),
                        ('Hamming', windows.hamming), ('Blackman', windows.blackman)]:
    w = win_func(N)
    axes[0].plot(w, label=name)

    # Réponse fréquentielle
    W = np.abs(fft(w, 4096))
    W_db = 20 * np.log10(W / W.max() + 1e-10)
    freq = np.linspace(0, 0.5, len(W_db)//2)
    axes[1].plot(freq[:500], W_db[:500], label=name)

axes[0].set_title('Fenêtres'); axes[0].legend()
axes[1].set_title('Réponse en fréquence (dB)'); axes[1].legend()
axes[1].set_ylim(-100, 5); axes[1].grid()
plt.tight_layout(); plt.show()

### Astuce : welch vs periodogram

`scipy.signal.welch` produit un estimé spectral plus lisse que le simple périodogramme
en moyennant les spectres de segments qui se chevauchent.

In [ ]:
from scipy.signal import welch, periodogram
import numpy as np
import matplotlib.pyplot as plt

fs = 1000
t = np.linspace(0, 2, fs*2)
signal = np.sin(2*np.pi*50*t) + 0.5*np.sin(2*np.pi*120*t) + np.random.randn(len(t))

f_per, Pxx_per = periodogram(signal, fs)
f_welch, Pxx_welch = welch(signal, fs, nperseg=256)

plt.figure(figsize=(10, 4))
plt.semilogy(f_per, Pxx_per, alpha=0.5, label='periodogram')
plt.semilogy(f_welch, Pxx_welch, 'r-', lw=2, label='welch')
plt.xlabel('Fréquence (Hz)'); plt.ylabel('DSP')
plt.legend(); plt.grid(); plt.xlim(0, 200); plt.show()

---

## Ressources externes

- [scipy.signal — documentation](https://docs.scipy.org/doc/scipy/reference/signal.html)
- [Digital Signal Processing — The Scientist and Engineer's Guide](https://www.analog.com/en/education/education-library/scientist_engineers_guide.html)